# 08 — Explainable Conversational AI Interface (RAG + Control)

The Phase 2 explainability layer. An operator can ask, in natural language,
**why** the RL agent made its scaling decisions, and receive plain-English
answers grounded in the agent's actual decision log (Retrieval-Augmented
Generation). The same chat interface also accepts operator **commands** —
surge warnings that set the agent's hint slots (function calling).

Pipeline: decision log -> readable documents -> embeddings (all-MiniLM-L6-v2)
-> FAISS retrieval -> Mistral-7B (via Ollama) grounded generation -> Gradio UI.

**Requires Ollama running (`ollama serve`) with the `mistral` model pulled.**

Uses `env.py`, `agent.py`.

## 1. Generate a decision log from the trained agent

In [1]:
import json
import numpy as np
import torch

from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic

stats = json.load(open('trace_params.json'))['stats']

net = ActorCritic()
net.load_state_dict(torch.load('ppo_sla-focused.pth'))
net.eval()

# run the canonical agent through one week, capturing a rich decision log
env = CloudClusterEnv(stats, seed=123)
obs, _ = env.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

decision_log = []
for t in range(STEPS_PER_WEEK):
    day, hour = env._current_day_hour()
    avg_cpu = env.history[-1][0]; avg_mem = env.history[-1][2]
    queue_before = len(env.queue); vms_before = env.active_vms
    with torch.no_grad():
        mean, _ = net.forward(obs_t.unsqueeze(0))
    obs, reward, done, tr, info = env.step(mean.squeeze(0).numpy())
    obs_t = torch.tensor(obs, dtype=torch.float32)
    vms_after = info['active_vms']; delta = vms_after - vms_before
    decision = (f"scaled UP by {delta}" if delta > 0
                else f"scaled DOWN by {abs(delta)}" if delta < 0 else "held steady")
    decision_log.append({
        'step': t, 'day': day, 'hour': hour,
        'cpu_load': round(float(avg_cpu),3), 'mem_load': round(float(avg_mem),3),
        'queue_before': queue_before, 'vms_before': vms_before,
        'vms_after': vms_after, 'decision': decision,
        'breaches': info['breaches'], 'cost': round(info['cost'],3),
        'utilisation': round(info['utilisation'],3)})
    if done: break

json.dump(decision_log, open('decision_log.json', 'w'), indent=2)
print(f"Captured {len(decision_log)} decisions. Saved decision_log.json")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Captured 672 decisions. Saved decision_log.json


## 2. Convert decisions to readable documents

In [2]:
def entry_to_text(e):
    return (f"On day {e['day']}, hour {e['hour']} (step {e['step']}): "
            f"CPU load was {e['cpu_load']:.0%}, memory load {e['mem_load']:.0%}, "
            f"with {e['queue_before']} jobs waiting in the queue. "
            f"The agent was running {e['vms_before']} VMs and {e['decision']}, "
            f"resulting in {e['vms_after']} VMs. "
            f"This step had {e['breaches']} SLA breaches, "
            f"cost {e['cost']:.2f}, and {e['utilisation']:.0%} utilisation.")

documents = [entry_to_text(e) for e in decision_log]
print(f"Created {len(documents)} documents. Example:")
print(" •", documents[1])

Created 672 documents. Example:
 • On day 0, hour 0 (step 1): CPU load was 100%, memory load 90%, with 336 jobs waiting in the queue. The agent was running 2 VMs and scaled UP by 5, resulting in 7 VMs. This step had 0 SLA breaches, cost 0.35, and 100% utilisation.


## 3. Embed documents and build the FAISS retrieval index

In [3]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(documents, show_progress_bar=True)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(doc_embeddings).astype('float32'))
print(f"FAISS index built with {index.ntotal} documents ({dim}-dim embeddings).")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

FAISS index built with 672 documents (384-dim embeddings).


## 4. RAG generation (retrieve + Mistral grounded answer)

In [4]:
import ollama

def answer_question(question, k=5):
    q_vec = embed_model.encode([question]).astype('float32')
    _, idxs = index.search(q_vec, k=k)
    retrieved = [documents[i] for i in idxs[0]]
    context = "\n".join(f"- {d}" for d in retrieved)
    prompt = (f"You are an assistant that explains an AI cloud-scaling agent's decisions.\n"
              f"Answer the question using ONLY the decision records below. "
              f"Do not invent information.\n\n"
              f"Decision records:\n{context}\n\nQuestion: {question}\n\nAnswer:")
    resp = ollama.chat(model='mistral', messages=[{'role':'user','content':prompt}])
    return resp['message']['content'], retrieved

# capture a few example Q&A for the report
example_questions = [
    "Why did the agent scale up on day 3?",
    "Were there any SLA breaches this week?",
    "What did the agent do during quiet periods?",
]
qa_examples = []
for q in example_questions:
    ans, srcs = answer_question(q)
    qa_examples.append({'question': q, 'answer': ans, 'sources': srcs[:3]})
    print(f"Q: {q}\nA: {ans}\n{'-'*60}")

json.dump(qa_examples, open('rag_qa_examples.json', 'w'), indent=2)
print("Saved rag_qa_examples.json (for the report)")

Q: Why did the agent scale up on day 3?
A:  The AI cloud-scaling agent scaled up on day 3 at step 300 because the CPU load was 100%, memory load was 75%, and there were 98 jobs waiting in the queue while running 3 VMs. This situation exceeded the current capacity, so the agent added 2 more VMs to meet the demand, resulting in a total of 5 VMs (as per the decision record provided).
------------------------------------------------------------
Q: Were there any SLA breaches this week?
A:  Yes, there were SLA breaches this week. Specifically, on day 6 at hour 14 (steps 634 and 635), a total of 52 + 13 = 65 SLA breaches occurred.
------------------------------------------------------------
Q: What did the agent do during quiet periods?
A:  During the quiet periods, as indicated by the CPU and memory loads being less than 100%, the agent maintained a steady state with a lower number of Virtual Machines (VMs). Specifically, on day 0 at hour 13 (step 52), the agent was running 2 VMs, and on da

## 5. Function-calling control — routing + hint tool

In [5]:
import re

tools = [
    {'type':'function','function':{'name':'explain_decision',
        'description':'Explain why the agent made a decision, or answer any question '
                      'about its past behaviour, load, VMs, breaches, or cost.',
        'parameters':{'type':'object','properties':{
            'question':{'type':'string','description':"operator's question"}},
            'required':['question']}}},
    {'type':'function','function':{'name':'set_surge_hint',
        'description':'Warn the agent a traffic surge is expected so it can scale up '
                      'in advance. Use for upcoming spikes/surges/increased load.',
        'parameters':{'type':'object','properties':{
            'hour':{'type':'integer','description':'hour (0-23) of the surge'},
            'magnitude':{'type':'string','description':'small, medium, or large'}},
            'required':['hour','magnitude']}}},
]

SYSTEM_PROMPT = ("You are a control interface for an autonomous cloud-scaling agent. "
    "You MUST respond by calling exactly one tool. Never answer directly in plain text.\n"
    "- Questions about past behaviour/decisions/load/VMs/breaches/cost -> explain_decision.\n"
    "- Warnings about an upcoming surge/spike/increased traffic -> set_surge_hint.\n"
    "When extracting the hour, convert to 24-hour format (2pm = 14, 9am = 9).\n"
    "Always call a tool.")

def route_message(msg):
    r = ollama.chat(model='mistral',
        messages=[{'role':'system','content':SYSTEM_PROMPT},
                  {'role':'user','content':msg}], tools=tools)
    m = r['message']
    if m.get('tool_calls'):
        tc = m['tool_calls'][0]
        return tc['function']['name'], tc['function']['arguments']
    text = m.get('content','')
    for pat in [r'\{.*"name".*\}', r'\[\s*\{.*"name".*\}\s*\]']:
        mt = re.search(pat, text, re.DOTALL)
        if mt:
            try:
                p = json.loads(mt.group())
                if isinstance(p, list): p = p[0]
                if 'name' in p: return p['name'], p.get('arguments', {})
            except Exception: pass
    return None, text

operator_hints = []
def handle_message(msg):
    name, args = route_message(msg)
    if name == 'set_surge_hint':
        hour = int(np.clip(args.get('hour',0), 0, 23))
        mag = str(args.get('magnitude','medium'))
        operator_hints.append({'hour':hour,'magnitude':mag})
        return (f"✓ Surge hint registered: {mag} surge at hour {hour}. "
                f"The agent will scale up in advance. (Active hints: {len(operator_hints)})")
    else:
        ans, _ = answer_question(args.get('question', msg) if isinstance(args, dict) else msg)
        return ans

# demonstrate both paths for the report
print("CONTROL TEST 1 (question):")
print(handle_message("Why did the agent scale up on day 3?"))
print("\nCONTROL TEST 2 (surge command):")
print(handle_message("Expect a large surge at 2pm"))

CONTROL TEST 1 (question):
 The AI cloud-scaling agent scaled up on day 3 at step 300 because the CPU load was 100%, memory load was 75%, and there were 98 jobs waiting in the queue. At this time, the agent was running 3 VMs and decided to scale UP by 2, resulting in 5 VMs. This decision was made to address the high CPU load, large number of waiting jobs, and aim for a more efficient utilization of resources.

CONTROL TEST 2 (surge command):
✓ Surge hint registered: large surge at hour 14. The agent will scale up in advance. (Active hints: 1)


## 6. Unified Gradio chat interface (explain + control)

In [6]:
import gradio as gr

def chat_fn(message, history):
    return handle_message(message)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Cloud Scaling Agent — Explain & Control Interface",
    description="Ask why the agent made decisions, or warn it about upcoming surges.",
    examples=["Why did the agent scale up on day 3?",
              "Were there any SLA breaches this week?",
              "Expect a large surge at 2pm",
              "When did the agent use the most VMs?"])
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
